# Where2Go DSS — lõi sản phẩm dùng chung

Notebook này gọi cùng catalog, ranker và planner với API/web. Dữ liệu legacy và evaluation cũ được giữ tại `notebooks/archive/legacy_poi_recommendation_system.ipynb` chỉ để đối chiếu lịch sử; không dùng các metric cũ làm bằng chứng chất lượng.

Chạy `scripts/build_catalog.py` và `scripts/setup_osrm.py` trước. Lịch trình dưới đây là case study, chưa có nhãn người dùng độc lập.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'where2go').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from where2go.catalog import load_catalog, coverage
from where2go.models import ItineraryRequest
from where2go.ranking import weights, CRITERIA
from where2go.routing import OSRM
from where2go.planner import plan_itinerary
from where2go.evaluation import metrics
from pprint import pprint
def display(*values):
    for value in values:
        pprint(value)
pois, manifest = load_catalog()
display(manifest)
display([row for row in coverage(pois) if row['location'] in ('Hà Nội', 'Đà Nẵng')])

## Trọng số và lịch trình

Ba tiêu chí: phù hợp sở thích (benefit), thời gian đường bộ (cost), độ tin cậy dữ liệu (benefit). Ma trận mặc định là so sánh bằng nhau; không phải ma trận chuyên gia. Fuzzy AHP dùng khoảng nhân đối xứng; với gamma chung, trọng số giải mờ có thể bằng crisp geometric mean. Đọc `docs/huong_dan_trien_khai.md` để xem giới hạn.

In [ ]:
request = ItineraryRequest(start={'latitude': 16.0612, 'longitude': 108.2227}, date='2026-09-20', location='Đà Nẵng', interests=['văn hóa', 'lịch sử'])
w, cr = weights(request.pairwise_preferences)
display(dict(zip(CRITERIA, w)), {'CR': cr})
result = plan_itinerary(pois, manifest, request, OSRM())
display({k: v for k, v in result.items() if k not in ('geometry', 'ranked_candidates')})

## Đánh giá đúng pipeline

Chạy `python scripts/evaluate_itineraries.py` để so bốn phương pháp trên cùng kịch bản/pool/planner. Chỉ truyền `--judgments` khi đã có nhãn độc lập. Metric dùng toàn tập được gán nhãn; không tạo query từ đáp án. Ví dụ tiếp theo chỉ là fixture kiểm tra công thức.

In [ ]:
display(metrics(['a'], {'a': 1, 'b': 1, 'c': 1}, k=10))
assert metrics(['a'], {'a': 1, 'b': 1, 'c': 1})['ndcg'] < 1